# Task 3: Car Price Prediction
Objective: Build a regression model that predicts the selling price of a used car based on features such as brand, age, mileage, fuel type, and transmission.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

### Load the Dataset
Please ensure `car_data.csv` (Vehicle dataset from cardekho) is in the same directory.

In [ ]:
try:
    df = pd.read_csv("car_data.csv")
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Error: Please download 'Vehicle dataset from cardekho' from Kaggle and place it as 'car_data.csv'.")
    # Creating dummy data
    data = {"Car_Name": ["Maruti Swift", "Hyundai i20", "Honda City", "Toyota Fortuner", "Hyundai Creta"] * 20,
            "Year": np.random.randint(2010, 2023, 100),
            "Selling_Price": np.random.uniform(2, 25, 100),
            "Present_Price": np.random.uniform(3, 30, 100),
            "Kms_Driven": np.random.randint(10000, 100000, 100),
            "Fuel_Type": np.random.choice(["Petrol", "Diesel", "CNG"], 100),
            "Seller_Type": np.random.choice(["Dealer", "Individual"], 100),
            "Transmission": np.random.choice(["Manual", "Automatic"], 100),
            "Owner": np.random.choice([0, 1, 2], 100)}
    df = pd.DataFrame(data)

### Data Cleaning & Feature Engineering

In [ ]:
print("Shape:", df.shape)
print("Nulls:\n", df.isnull().sum())
df.drop_duplicates(inplace=True)

# Calculate car age (assuming current year is 2024)
df["Car_Age"] = 2024 - df["Year"]

# Extract brand from Car_Name (first word)
df["Brand"] = df["Car_Name"].apply(lambda x: str(x).split(" ")[0])
df.head()

### Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["Selling_Price"], kde=True)
plt.title("Distribution of Selling Prices")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Fuel_Type", y="Selling_Price")
plt.title("Selling Price vs Fuel Type")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Car_Age", y="Selling_Price")
plt.title("Selling Price vs Car Age")
plt.show()

### Encoding Categorical Variables

In [ ]:
categorical_cols = ["Fuel_Type", "Seller_Type", "Transmission", "Brand"]
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Drop original columns we extracted from
df.drop(["Car_Name", "Year"], axis=1, inplace=True)

### Feature Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

### Train/Test Split

In [ ]:
X = df.drop("Selling_Price", axis=1)
y = df["Selling_Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Train Models

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)

### Evaluate Models

In [ ]:
def evaluate(name, y_true, y_pred):
    print(f"--- {name} ---")
    print(f"MAE:  {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
    print(f"R2:   {r2_score(y_true, y_pred):.4f}\n")

evaluate("Linear Regression", y_test, lr_pred)
evaluate("Random Forest Regressor", y_test, rf_pred)
evaluate("Gradient Boosting", y_test, gb_pred)

### Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values().plot(kind="barh", color="teal")
plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance Score")
plt.show()